# Chapter 3 Lab — The Frequency-Domain View

**Think of it like this:** that squiggly EEG line is actually several instruments playing *at the same time*, mixed into one sound — like a band recorded through a single microphone. You can't tell by ear alone "how loud is the bass right now" from the mixed track. A **spectrogram** is a tool that unmixes the recording back into its separate instruments (frequency bands) over time, so you can see "the bass (delta) just got loud" or "the triangle (beta) is busy right now."

The five "instruments" in EEG:
- **Delta** (slow, deep) — dominates N3, the deepest sleep
- **Theta** — shows up in N1 and REM
- **Alpha** — relaxed wakefulness, eyes closed
- **Sigma** (the spindle band) — N2's little "ding" bursts
- **Beta** (fast, busy) — alert wakefulness

**Goal of this lab:** compute a spectrogram across a whole night and literally watch the instruments change as sleep stage changes.

In [ ]:
!pip install -q mne
%matplotlib inline
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.datasets.sleep_physionet.age import fetch_data

ALICE = 0
alice_files = fetch_data(subjects=[ALICE], recording=[1])[0]
raw = mne.io.read_raw_edf(
    alice_files[0], stim_channel="Event marker", infer_types=True, preload=True, verbose="error",
)
annot = mne.read_annotations(alice_files[1])
raw.set_annotations(annot, emit_warning=False)
annot.crop(annot[1]["onset"] - 30 * 60, annot[-2]["onset"] + 30 * 60)
raw.set_annotations(annot, emit_warning=False)

## Compute the spectrogram

We'll use one EEG channel across the whole cropped night. `raw.compute_psd` with a sliding window gives us power-over-time-and-frequency — exactly a spectrogram.

In [ ]:
eeg_channel = [ch for ch in raw.ch_names if "EEG" in ch][0]
print("Using channel:", eeg_channel)

spectrum = raw.compute_psd(
    method="welch", picks=[eeg_channel], fmin=0.5, fmax=30.0,
    n_fft=int(raw.info["sfreq"] * 30), n_overlap=0, verbose="error",
)

That gave us one average spectrum for the whole night. To see it change *over time*, we instead slide a 30-second window across the recording and compute a spectrum for each window — one column per epoch.

In [ ]:
sfreq = raw.info["sfreq"]
win_samples = int(30 * sfreq)
data = raw.get_data(picks=[eeg_channel])[0]
n_windows = len(data) // win_samples

from scipy.signal import welch

spec_cols = []
freqs = None
for i in range(n_windows):
    seg = data[i * win_samples:(i + 1) * win_samples]
    f, p = welch(seg, fs=sfreq, nperseg=int(sfreq * 4))
    freqs = f
    spec_cols.append(p)

spectrogram = np.array(spec_cols).T  # shape: (freq bins, time windows)
print("Spectrogram shape:", spectrogram.shape, "(frequency bins x 30-second epochs)")

## Plot it, with the hypnogram underneath

Top panel: the spectrogram (brighter = more power at that frequency, at that point in the night). Bottom panel: the expert's actual sleep-stage labels for the same stretch of time. Look for the match — delta (bottom of the frequency axis) should glow brightest exactly where the hypnogram says N3.

In [ ]:
annotation_desc_2_event_id = {
    "Sleep stage W": 1, "Sleep stage 1": 2, "Sleep stage 2": 3,
    "Sleep stage 3": 4, "Sleep stage 4": 4, "Sleep stage R": 5,
}
events, _ = mne.events_from_annotations(raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0)
stage_per_epoch = events[:n_windows, 2]
stage_names = {1: "Wake", 2: "N1", 3: "N2", 4: "N3", 5: "REM"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax1.pcolormesh(np.arange(n_windows), freqs, 10 * np.log10(spectrogram + 1e-12), shading="auto", cmap="magma")
ax1.set_ylabel("Frequency (Hz)")
ax1.set_ylim(0, 30)
ax1.set_title("Spectrogram (brighter = more power)")

ax2.step(np.arange(len(stage_per_epoch)), stage_per_epoch, where="post")
ax2.set_yticks(list(stage_names.keys()))
ax2.set_yticklabels(list(stage_names.values()))
ax2.invert_yaxis()
ax2.set_xlabel("30-second epoch number")
ax2.set_title("Expert's hypnogram")

plt.tight_layout()
plt.show()

## Done when...

You can point at a spot on the spectrogram and guess the sleep stage **before** checking the hypnogram panel underneath it — because you can see delta glowing during N3, and the pattern thinning out and lifting during Wake and REM.

That's Part 1 done. Chapter 4 turns this same idea (band power per epoch) into actual numbers a classifier can learn from.